In [ ]:
# Instala ou atualiza silenciosamente (-q) o Ultralytics (YOLO), PyTorch e PyYAML
!pip -q install -U ultralytics torch pyyaml

# Importa biblioteca para buscar arquivos usando padrões de caminho (ex: *.zip, *.pt)
import glob
# Importa biblioteca para ler e manipular arquivos no formato JSON (anotações COCO)
import json
# Importa biblioteca para manipular caminhos, pastas e variáveis do sistema operacional
import os
# Importa biblioteca para copiar e apagar arquivos e diretórios inteiros
import shutil
# Importa biblioteca para descompactar arquivos .zip enviados do Roboflow
import zipfile
# Importa a classe Path para manipular nomes e extensões de arquivos de forma moderna
from pathlib import Path

# Importa o PyTorch, usado aqui para criar o agendador ReduceLROnPlateau
import torch
# Importa o PyYAML para ler e gravar arquivos de configuração data.yaml
import yaml
# Importa o utilitário do Google Colab para fazer upload e download de arquivos
from google.colab import files
# Importa funções do IPython para exibir imagens (gráficos e matriz) direto na célula
from IPython.display import Image, display
# Importa a classe principal YOLO da biblioteca Ultralytics
from ultralytics import YOLO

In [ ]:
# Define o modelo pré-treinado base (fora da versão nano: 'yolo26s.pt' ou 'yolo26m.pt')
MODELO_YOLO = "yolo26s.pt"
# Define o número total de épocas (passagens completas pelo dataset) no treino
EPOCAS = 50
# Define quantas imagens são processadas por vez antes de atualizar os pesos
BATCH_SIZE = 8
# Define a resolução (640x640 pixels) para redimensionar as imagens no treino
IMG_SIZE = 640
# Define a confiança mínima (10%) para aceitar uma detecção no vídeo de teste
CONF_INFERENCIA = 0.10
# Define o limiar de sobreposição (IoU) para eliminar caixas duplicadas no mesmo objeto
IOU_INFERENCIA = 0.45


# Função que valida e carrega o modelo YOLO escolhido
def carregar_modelo(nome_modelo: str) -> YOLO:
    # Bloqueia o uso da versão nano caso o nome termine com 'n.pt'
    if nome_modelo.endswith("n.pt"):
        # Lança um erro informando que apenas versões Small ou Medium são permitidas
        raise ValueError("Versão nano bloqueada. Use 'yolo26s.pt' ou 'yolo26m.pt'.")
    # Retorna o objeto YOLO carregado com os pesos pré-treinados escolhidos
    return YOLO(nome_modelo)


# Função que extrai o ZIP do Roboflow e organiza as 31 fotos em treino e validação
def preparar_dataset_31_fotos(
    zip_path: str,
    extract_dir: str = "/content/raw_dataset",
    out_dir: str = "/content/dataset_31",
):
    # Remove pastas temporárias anteriores para evitar mistura de arquivos antigos
    shutil.rmtree(extract_dir, ignore_errors=True)
    # Remove a pasta final anterior do dataset, caso já exista
    shutil.rmtree(out_dir, ignore_errors=True)
    # Cria o diretório vazio onde o ZIP será extraído
    os.makedirs(extract_dir, exist_ok=True)

    # Abre o arquivo ZIP em modo de leitura
    with zipfile.ZipFile(zip_path, "r") as zf:
        # Extrai todo o conteúdo do ZIP para dentro de extract_dir
        zf.extractall(extract_dir)

    # Cria as subpastas 'images' e 'labels' tanto para 'train' quanto para 'valid'
    for split in ["train", "valid"]:
        os.makedirs(os.path.join(out_dir, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(out_dir, split, "labels"), exist_ok=True)

    # Procura arquivos data.yaml (padrão YOLO) dentro da pasta extraída
    yaml_files = glob.glob(os.path.join(extract_dir, "**", "data.yaml"), recursive=True)
    # Procura arquivos .json (padrão COCO) dentro da pasta extraída
    coco_files = glob.glob(os.path.join(extract_dir, "**", "*.json"), recursive=True)

    # Caso o dataset esteja no formato YOLO (com data.yaml)
    if yaml_files:
        # Pega o caminho do primeiro data.yaml encontrado
        src_yaml = yaml_files[0]
        # Obtém a pasta raiz onde o data.yaml está localizado
        src_root = os.path.dirname(src_yaml)
        # Abre e lê o conteúdo do data.yaml original
        with open(src_yaml, "r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)

        # Extrai a lista de nomes das classes (usa 'skate-de-dedo' como padrão)
        class_names = cfg.get("names", ["skate-de-dedo"])
        # Se as classes vierem como dicionário {0: 'classe'}, converte para lista ordenada
        if isinstance(class_names, dict):
            class_names = [class_names[k] for k in sorted(class_names.keys())]

        # Lista todas as imagens (.jpg, .jpeg, .png, .webp) encontradas no dataset
        img_paths = sorted([
            p for p in glob.glob(os.path.join(src_root, "**", "*.*"), recursive=True)
            if Path(p).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
        ])
        # Mapeia cada arquivo de anotação .txt pelo nome base (sem extensão)
        lbl_map = {
            Path(p).stem: p
            for p in glob.glob(os.path.join(src_root, "**", "*.txt"), recursive=True)
            if Path(p).name.lower() != "readme.txt"
        }

        # Percorre cada imagem encontrada para copiá-la para train e valid
        for img_p in img_paths:
            # Extrai o nome do arquivo sem extensão
            stem = Path(img_p).stem
            # Extrai a extensão original do arquivo (ex: .jpg)
            ext = Path(img_p).suffix
            # Replica a imagem e sua respectiva label em 'train' e 'valid'
            for split in ["train", "valid"]:
                # Copia a imagem para a pasta de destino
                shutil.copy2(img_p, os.path.join(out_dir, split, "images", f"{stem}{ext}"))
                # Define o caminho de saída do arquivo .txt de anotação
                lbl_out = os.path.join(out_dir, split, "labels", f"{stem}.txt")
                # Se existir anotação para a imagem, copia o arquivo .txt
                if stem in lbl_map:
                    shutil.copy2(lbl_map[stem], lbl_out)
                # Caso contrário (imagem negativa sem objeto), cria um .txt vazio
                else:
                    Path(lbl_out).write_text("")

    # Caso o dataset esteja no formato COCO JSON
    elif coco_files:
        # Abre e carrega o arquivo JSON de anotações COCO
        with open(coco_files[0], "r", encoding="utf-8") as f:
            coco = json.load(f)

        # Mapeia IDs de categoria do COCO para seus respectivos nomes
        cat_map = {c["id"]: c["name"] for c in coco["categories"]}
        # Coleta os IDs únicos de categorias realmente usadas nas anotações
        used_ids = sorted({a["category_id"] for a in coco["annotations"]})
        # Reindexa os IDs para começarem em 0, 1, 2... conforme exigido pelo YOLO
        id_to_idx = {cid: idx for idx, cid in enumerate(used_ids)}
        # Monta a lista final de nomes das classes ordenadas
        class_names = [cat_map[cid] for cid in used_ids]

        # Agrupa todas as bounding boxes por image_id
        anns_by_img = {}
        for ann in coco["annotations"]:
            anns_by_img.setdefault(ann["image_id"], []).append(ann)

        # Indexa todas as imagens extraídas pelo nome do arquivo
        all_imgs = {
            os.path.basename(p): p
            for p in glob.glob(os.path.join(extract_dir, "**", "*.*"), recursive=True)
            if Path(p).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
        }

        # Converte as anotações de cada imagem do formato COCO para o formato YOLO
        for img_info in coco["images"]:
            # Obtém o nome do arquivo da imagem
            fname = os.path.basename(img_info["file_name"])
            # Busca o caminho completo da imagem extraída
            src_img = all_imgs.get(fname)
            # Pula se a imagem não for encontrada na pasta
            if not src_img:
                continue

            # Obtém largura e altura originais da imagem em pixels
            w_img, h_img = img_info["width"], img_info["height"]
            # Extrai o nome base sem extensão
            stem = Path(fname).stem
            # Lista que armazenará as linhas de anotação YOLO desta imagem
            linhas = []
            # Converte cada caixa [x_canto, y_canto, largura, altura] para centro normalizado (0 a 1)
            for ann in anns_by_img.get(img_info["id"], []):
                x, y, bw, bh = ann["bbox"]
                xc = (x + bw / 2.0) / w_img
                yc = (y + bh / 2.0) / h_img
                nw = bw / w_img
                nh = bh / h_img
                cls_idx = id_to_idx[ann["category_id"]]
                linhas.append(f"{cls_idx} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

            # Salva a imagem e o arquivo .txt convertido em 'train' e 'valid'
            for split in ["train", "valid"]:
                shutil.copy2(src_img, os.path.join(out_dir, split, "images", fname))
                Path(os.path.join(out_dir, split, "labels", f"{stem}.txt")).write_text("\n".join(linhas))
    # Se não houver nem data.yaml nem JSON COCO, interrompe com erro claro
    else:
        raise FileNotFoundError("Não foi encontrado data.yaml nem arquivo JSON COCO no ZIP.")

    # Define o caminho onde o novo data.yaml unificado será salvo
    out_yaml = os.path.join(out_dir, "data.yaml")
    # Monta o dicionário de configuração apontando para train/images e valid/images
    novo_cfg = {
        "path": out_dir,
        "train": "train/images",
        "val": "valid/images",
        "nc": len(class_names),
        "names": class_names,
    }
    # Grava o novo arquivo data.yaml no disco
    with open(out_yaml, "w", encoding="utf-8") as f:
        yaml.dump(novo_cfg, f, allow_unicode=True)

    # Conta quantas imagens ficaram em treino e validação para confirmação visual
    qtd_train = len(glob.glob(os.path.join(out_dir, "train", "images", "*.*")))
    qtd_val = len(glob.glob(os.path.join(out_dir, "valid", "images", "*.*")))
    print(f"Dataset pronto | Classes: {class_names}")
    print(f"Imagens em train: {qtd_train} | Imagens em valid: {qtd_val}")
    # Retorna o caminho do data.yaml gerado e o dicionário de configuração
    return out_yaml, novo_cfg


# Função que acopla o ReduceLROnPlateau do PyTorch ao ciclo de treino do YOLO
def anexar_reduce_lr_on_plateau(trainer, mode="min", factor=0.5, patience=6, min_lr=1e-6):
    # Dicionário mutável para guardar a instância do scheduler entre as épocas
    estado = {"scheduler": None}

    # Callback executado automaticamente ao final de cada época de treino
    def ao_fim_da_epoca(t):
        # Garante que o otimizador já foi inicializado antes de agir
        if not hasattr(t, "optimizer") or t.optimizer is None:
            return
        # Na primeira época, instancia o ReduceLROnPlateau vinculado ao otimizador
        if estado["scheduler"] is None:
            estado["scheduler"] = torch.optim.lr_scheduler.ReduceLROnPlateau(
                t.optimizer,
                mode=mode,          # 'min' reduz o LR quando a perda para de cair
                factor=factor,      # Multiplica o LR por 0.5 quando estagnar
                patience=patience,  # Espera 6 épocas sem melhora antes de reduzir
                min_lr=min_lr,      # Limite mínimo para a taxa de aprendizado
            )

        # Recupera o dicionário de métricas de validação da época atual
        metrics = getattr(t, "metrics", {}) or {}
        # Extrai as perdas de caixa (box), classificação (cls) e distribuição (dfl)
        val_box = metrics.get("val/box_loss", 0.0)
        val_cls = metrics.get("val/cls_loss", 0.0)
        val_dfl = metrics.get("val/dfl_loss", 0.0)
        # Soma as três componentes para obter a perda total de validação
        perda_val = float(val_box) + float(val_cls) + float(val_dfl)

        # Registra a learning rate antes do passo do scheduler
        lr_antes = t.optimizer.param_groups[0]["lr"]
        # Atualiza o scheduler com a perda de validação atual
        estado["scheduler"].step(perda_val)
        # Registra a learning rate após a verificação
        lr_depois = t.optimizer.param_groups[0]["lr"]

        # Exibe um aviso no console sempre que a learning rate for reduzida
        if lr_depois < lr_antes:
            print(
                f"\n[ReduceLROnPlateau] LR reduzido: {lr_antes:.2e} -> {lr_depois:.2e} "
                f"(val_loss={perda_val:.4f})"
            )

    # Registra a função callback no evento de fim de época do treinador YOLO
    trainer.add_callback("on_fit_epoch_end", ao_fim_da_epoca)

In [ ]:
# Verifica se já existe algum arquivo .zip enviado na pasta /content do Colab
zips = glob.glob("/content/*.zip")
# Se não houver nenhum ZIP, abre o botão para você selecionar e enviar do computador
if not zips:
    print("Envie o arquivo .zip do Roboflow:")
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]
# Se já existir um ZIP no Colab, reutiliza-o automaticamente
else:
    zip_path = zips[0]
    print(f"Usando ZIP: {zip_path}")

# Executa a extração e estruturação das 31 fotos gerando o data.yaml
data_yaml, cfg = preparar_dataset_31_fotos(zip_path)

In [ ]:
# Carrega o modelo base escolhido (yolo26s.pt ou yolo26m.pt)
modelo = carregar_modelo(MODELO_YOLO)
# Anexa o agendador ReduceLROnPlateau ao treinador do modelo
anexar_reduce_lr_on_plateau(modelo, mode="min", factor=0.5, patience=6, min_lr=1e-6)

# Inicia o treinamento do modelo com data augmentation e parâmetros definidos
resultados = modelo.train(
    data=data_yaml,            # Caminho do arquivo data.yaml com as 31 imagens
    epochs=EPOCAS,             # Quantidade de épocas de treinamento
    imgsz=IMG_SIZE,            # Resolução de entrada das imagens (640x640)
    batch=BATCH_SIZE,          # Tamanho do lote (8 imagens por iteração)
    cos_lr=False,              # Desativa o agendador cosseno padrão para usar o ReduceLROnPlateau
    lr0=1e-3,                  # Taxa de aprendizado inicial
    lrf=1.0,                   # Mantém o fator final em 1.0 para não interferir no ReduceLROnPlateau
    degrees=20.0,              # Rotação aleatória de até ±20 graus (data augmentation)
    scale=0.5,                 # Variação de escala/zoom de até 50% (data augmentation)
    hsv_s=0.6,                 # Variação de saturação de cor (data augmentation)
    hsv_v=0.4,                 # Variação de brilho/iluminação (data augmentation)
    plots=True,                # Gera gráficos de perda, curvas e matriz de confusão
    project="/content/runs",   # Pasta raiz onde os resultados serão salvos
    name=f"treino_{Path(MODELO_YOLO).stem}_31fotos_plateau",  # Nome da subpasta do experimento
    exist_ok=True,             # Sobrescreve a pasta se já existir um treino com mesmo nome
)

# Salva o caminho completo da pasta onde este treino foi gravado
run_dir = str(resultados.save_dir)
# Carrega os melhores pesos salvos (best.pt) obtidos durante o treinamento
best_model = YOLO(os.path.join(run_dir, "weights", "best.pt"))
# Confirma no console o fim do treino e o diretório de saída
print("Treinamento finalizado. Pasta:", run_dir)

In [ ]:
# Executa a validação oficial do melhor modelo (best.pt) sobre o conjunto de validação
val_metrics = best_model.val()

# Imprime um resumo formatado com as principais métricas de detecção
print("=" * 55)
print(f"Modelo: {MODELO_YOLO} | Imagens usadas: 31 treino / 31 valid")
print(f"Precision: {val_metrics.box.mp:.3f} | Recall: {val_metrics.box.mr:.3f}")
print(f"mAP@50:    {val_metrics.box.map50:.3f} | mAP@50-95: {val_metrics.box.map:.3f}")
print("=" * 55)

# Exibe o gráfico de evolução das perdas (loss) e métricas ao longo das épocas
display(Image(filename=os.path.join(run_dir, "results.png"), width=850))
# Exibe a matriz de confusão gerada na avaliação
display(Image(filename=os.path.join(run_dir, "confusion_matrix.png"), width=550))

In [ ]:
# Busca todos os arquivos best.pt salvos na pasta runs para garantir o peso mais recente
pesos = sorted(glob.glob("/content/runs/**/weights/best.pt", recursive=True))
# Carrega o modelo treinado mais recente
best_model = YOLO(pesos[-1])

# Solicita o upload do vídeo de teste (em close-up) pelo navegador
print("Envie o vídeo de perto (.mp4/.avi):")
up_video = files.upload()
# Pega o nome do arquivo de vídeo recém-enviado
video_path = list(up_video.keys())[0]

# Remove a pasta de inferência anterior para evitar baixar um vídeo antigo por engano
shutil.rmtree("/content/runs/inferencia_closeup", ignore_errors=True)

# Roda a detecção frame a frame no vídeo enviado
pred = best_model.predict(
    source=video_path,          # Caminho do vídeo enviado
    conf=CONF_INFERENCIA,       # Limiar de confiança mínima (0.10)
    iou=IOU_INFERENCIA,         # Limiar de supressão de caixas sobrepostas (0.45)
    max_det=1,                  # Mantém apenas a detecção de maior confiança por frame
    save=True,                  # Salva o vídeo resultante com as caixas desenhadas
    project="/content/runs",    # Pasta raiz de saída
    name="inferencia_closeup",  # Subpasta onde o vídeo processado será salvo
    exist_ok=True,              # Permite reutilizar o nome da pasta
)

# Extrai a confiança de cada frame em que o skate foi detectado
confs = [float(r.boxes.conf[0]) for r in pred if len(r.boxes) > 0]
# Mostra quantos frames tiveram detecção e qual foi a maior confiança atingida
print(f"Frames com detecção: {len(confs)}/{len(pred)} | Confiança máxima: {max(confs) if confs else 0:.3f}")

# Localiza o arquivo de vídeo gerado na pasta de saída e inicia o download automático
saida = glob.glob(os.path.join(str(pred[0].save_dir), "*.*"))[0]
files.download(saida)